# Cleaning Data

**Prerequisites**

- [Intro](https://datascience.quantecon.org/intro.html)  
- [Boolean selection](https://datascience.quantecon.org/basics.html)  
- [Indexing](https://datascience.quantecon.org/the_index.html)  


**Outcomes**

- Be able to use string methods to clean data that comes as a string  
- Be able to drop missing data  
- Use cleaning methods to prepare and analyze a real dataset  


**Data**

- Item information from about 3,000 Chipotle meals from about 1,800
  Grubhub orders  

In [ ]:
# Uncomment following line to install on colab
#! pip install 

In [1]:
import pandas as pd
import numpy as np

## Cleaning Data

For many data projects, a [significant proportion of
time](https://www.forbes.com/sites/gilpress/2016/03/23/data-preparation-most-time-consuming-least-enjoyable-data-science-task-survey-says/#74d447456f63)
is spent collecting and cleaning the data — not performing the analysis.

This non-analysis work is often called “data cleaning”.

pandas provides very powerful data cleaning tools, which we
will demonstrate using the following dataset.

In [2]:
df = pd.DataFrame({"numbers": ["#23", "#24", "#18", "#14", "#12", "#10", "#35"],
                   "nums": ["23", "24", "18", "14", np.nan, "XYZ", "35"],
                   "colors": ["green", "red", "yellow", "orange", "purple", "blue", "pink"],
                   "other_column": [0, 1, 0, 2, 1, 0, 2]})
df

,numbers,nums,colors,other_column
0,#23,23,green,0
1,#24,24,red,1
2,#18,18,yellow,0
3,#14,14,orange,2
4,#12,NaN,purple,1
5,#10,XYZ,blue,0
6,#35,35,pink,2


What would happen if we wanted to try and compute the mean of
`numbers`?

In [3]:
df["numbers"].mean()

TypeError: Could not convert #23#24#18#14#12#10#35 to numeric

&rarr; Shows the error message, since the data type is not numeric, containing '#'

It throws an error!

Can you figure out why?

When looking at error messages, start at the very bottom.

The final error says, `TypeError: Could not convert #23#24... to numeric`.

## Exercise

See exercise 1 in the [exercise list](#pd-cln-ex).

## String Methods

Our solution to the previous exercise was to remove the `#` by using
the `replace` string method: `int(c2n.replace("#", ""))`.

One way to make this change to every element of a column would be to
loop through all elements of the column and apply the desired string
methods…

In [ ]:
%%time

# Iterate over all rows
for row in df.iterrows():

    # `iterrows` method produces a tuple with two elements...
    # The first element is an index and the second is a Series with the data from that row
    index_value, column_values = row

    # Apply string method
    clean_number = int(column_values["numbers"].replace("#", ""))

    # The `at` method is very similar to the `loc` method, but it is specialized
    # for accessing single elements at a time... We wanted to use it here to give
    # the loop the best chance to beat a faster method which we show you next.
    df.at[index_value, "numbers_loop"] = clean_number

While this is fast for a small dataset like this, this method slows for larger datasets.

One *significantly* faster (and easier) method is to apply a string
method to an entire column of data.

Most methods that are available to a Python string (we learned a
few of them in the [strings lecture](https://datascience.quantecon.org/../python_fundamentals/basics.html)) are
also available to a pandas Series that has `dtype` object.

We access them by doing `s.str.method_name` where `method_name` is
the name of the method.

When we apply the method to a Series, it is applied to all rows in the
Series in one shot!

Let’s redo our previous example using a pandas `.str` method.

In [ ]:
%%time

# ~2x faster than loop... However, speed gain increases with size of DataFrame. The
# speedup can be in the ballpark of ~100-500x faster for big DataFrames.
# See appendix at the end of the lecture for an application on a larger DataFrame
df["numbers_str"] = df["numbers"].str.replace("#", "")

We can use `.str` to access almost any string method that works on
normal strings. (See the [official
documentation](https://pandas.pydata.org/pandas-docs/stable/text.html)
for more information.)

In [ ]:
df["colors"].str.contains("p")

In [4]:
df["colors"].str.capitalize()

0     Green
1       Red
2    Yellow
3    Orange
4    Purple
5      Blue
6      Pink
Name: colors, dtype: object

## Exercise

See exercise 2  in the [exercise list](#pd-cln-ex).

## Type Conversions

In our example above, the `dtype` of the `numbers_str` column shows that pandas still treats
it as a string even after we have removed the `"#"`.

We need to convert this column to numbers.

The best way to do this is using the `pd.to_numeric` function.

This method attempts to convert whatever is stored in a Series into
numeric values

For example, after the `"#"` removed, the numbers of column
`"numbers"` are ready to be converted to actual numbers.

In [5]:
df["numbers_numeric"] = pd.to_numeric(df["numbers_str"])

KeyError: 'numbers_str'

In [ ]:
df.dtypes

In [ ]:
df.head()

We can convert to other types well.

Using the `astype` method, we can convert to any of the supported
pandas `dtypes` (recall the [intro lecture](https://datascience.quantecon.org/intro.html)).

Below are some examples. (Pay attention to the reported `dtype`)

In [ ]:
df["numbers_numeric"].astype(str)

In [ ]:
df["numbers_numeric"].astype(float)

## Exercise

See exercise 3 in the [exercise list](#pd-cln-ex).

## Missing Data

Many datasets have missing data.

In our example, we are missing an element from the `"nums"` column.

In [6]:
df

,numbers,nums,colors,other_column
0,#23,23,green,0
1,#24,24,red,1
2,#18,18,yellow,0
3,#14,14,orange,2
4,#12,NaN,purple,1
5,#10,XYZ,blue,0
6,#35,35,pink,2


We can find missing data by using the `isnull` method.

In [7]:
df.isnull()

,numbers,nums,colors,other_column
0,False,False,False,False
1,False,False,False,False
2,False,False,False,False
3,False,False,False,False
4,False,True,False,False
5,False,False,False,False
6,False,False,False,False


We might want to know whether particular rows or columns have any
missing data.

To do this we can use the `.any` method on the boolean DataFrame
`df.isnull()`.

In [8]:
df.isnull().any(axis=0)

numbers         False
nums             True
colors          False
other_column    False
dtype: bool

In [9]:
df.isnull().any(axis=1)

0    False
1    False
2    False
3    False
4     True
5    False
6    False
dtype: bool

Many approaches have been developed to deal with missing data, but the two most commonly used (and the corresponding DataFrame method) are:

- Exclusion: Ignore any data that is missing (`.dropna`).  
- Imputation: Compute “predicted” values for the data that is missing
  (`.fillna`).  


For the advantages and disadvantages of these (and other) approaches,
consider reading the [Wikipedia
article](https://en.wikipedia.org/wiki/Missing_data).

For now, let’s see some examples.

In [10]:
# drop all rows containing a missing observation
df.dropna()

,numbers,nums,colors,other_column
0,#23,23,green,0
1,#24,24,red,1
2,#18,18,yellow,0
3,#14,14,orange,2
5,#10,XYZ,blue,0
6,#35,35,pink,2


In [11]:
# fill the missing values with a specific value
df.fillna(value=100)

,numbers,nums,colors,other_column
0,#23,23,green,0
1,#24,24,red,1
2,#18,18,yellow,0
3,#14,14,orange,2
4,#12,100,purple,1
5,#10,XYZ,blue,0
6,#35,35,pink,2


In [12]:
# use the _next_ valid observation to fill the missing data
df.fillna(method="bfill")

,numbers,nums,colors,other_column
0,#23,23,green,0
1,#24,24,red,1
2,#18,18,yellow,0
3,#14,14,orange,2
4,#12,XYZ,purple,1
5,#10,XYZ,blue,0
6,#35,35,pink,2


In [13]:
# use the _previous_ valid observation to fill missing data
df.fillna(method="ffill")

,numbers,nums,colors,other_column
0,#23,23,green,0
1,#24,24,red,1
2,#18,18,yellow,0
3,#14,14,orange,2
4,#12,14,purple,1
5,#10,XYZ,blue,0
6,#35,35,pink,2


We will see more examples of dealing with missing data in future
chapters.

## Case Study

We will now use data from an
[article](https://www.nytimes.com/interactive/2015/02/17/upshot/what-do-people-actually-order-at-chipotle.html)
written by The Upshot at the NYTimes.

This data has order information from almost 2,000 Chipotle orders and
includes information on what was ordered and how much it cost.

In [14]:
url = "https://datascience.quantecon.org/assets/data/chipotle_raw.csv.zip"
chipotle = pd.read_csv(url)
chipotle.head()

,Unnamed: 0,order_id,quantity,item_name,choice_description,item_price
0,0,1,1,Chips and Fresh Tomato Salsa,NaN,$2.39
1,1,1,1,Izze,[Clementine],$3.39
2,2,1,1,Nantucket Nectar,[Apple],$3.39
3,3,1,1,Chips and Tomatillo-Green Chili Salsa,NaN,$2.39
4,4,2,2,Chicken Bowl,"[Tomatillo-Red Chili Salsa (Hot), [Black Beans...",$16.98


## Exercise

See exercise 4 in the [exercise list](#pd-cln-ex).

## Appendix: Performance of `.str` Methods

Let’s repeat the “remove the `#`” example from above, but this time on
a much larger dataset.

In [15]:
import numpy as np
test = pd.DataFrame({"floats": np.round(100*np.random.rand(100000), 2)})
test["strings"] = test["floats"].astype(str) + "%"
test.head()

,floats,strings
0,95.72,95.72%
1,6.91,6.91%
2,43.43,43.43%
3,93.76,93.76%
4,72.34,72.34%


In [16]:
%%time

for row in test.iterrows():
    index_value, column_values = row
    clean_number = column_values["strings"].replace("%", "")
    test.at[index_value, "numbers_loop"] = clean_number

CPU times: user 1.32 s, sys: 10 ms, total: 1.33 s
Wall time: 1.33 s


In [17]:
%%time
test["numbers_str_method"] = test["strings"].str.replace("%", "")

CPU times: user 10.6 ms, sys: 1.03 ms, total: 11.6 ms
Wall time: 11 ms


In [18]:
test["numbers_str_method"].equals(test["numbers_loop"])

True

We got the exact same result in a fraction of the time!


<a id='pd-cln-ex'></a>

## Exercises

### Exercise 1

Convert the string below into a number.

In [19]:
c2n = "#39"

([back to text](#pd-cln-dir1))

### Exercise 2

Make a new column called `colors_upper` that contains the elements of
`colors` with all uppercase letters.

([back to text](#pd-cln-dir2))

In [20]:
df

,numbers,nums,colors,other_column
0,#23,23,green,0
1,#24,24,red,1
2,#18,18,yellow,0
3,#14,14,orange,2
4,#12,NaN,purple,1
5,#10,XYZ,blue,0
6,#35,35,pink,2


In [21]:
df['colors_upper'] = df['colors'].str.upper()
df.head()

,numbers,nums,colors,other_column,colors_upper
0,#23,23,green,0,GREEN
1,#24,24,red,1,RED
2,#18,18,yellow,0,YELLOW
3,#14,14,orange,2,ORANGE
4,#12,NaN,purple,1,PURPLE


### Exercise 3

Convert the column `"nums"` to a numeric type using `pd.to_numeric` and
save it to the DataFrame as `"nums_tonumeric"`.

Notice that there is a missing value, and a value that is not a number.

Look at the documentation for `pd.to_numeric` and think about how to
overcome this.

Think about why this could be a bad idea of used without
knowing what your data looks like. (Think about what happens when you
apply it to the `"numbers"` column before replacing the `"#"`.)

([back to text](#pd-cln-dir3))

In [22]:
pd.to_numeric?

**The documentation result for the `pd.to_numeric` is as follows:**  
  
Signature: pd.to_numeric(arg, errors='raise', downcast=None)
Docstring:
Convert argument to a numeric type.

The default return dtype is `float64` or `int64`
depending on the data supplied. Use the `downcast` parameter
to obtain other dtypes.

Please note that precision loss may occur if really large numbers
are passed in. Due to the internal limitations of `ndarray`, if
numbers smaller than `-9223372036854775808` (np.iinfo(np.int64).min)
or larger than `18446744073709551615` (np.iinfo(np.uint64).max) are
passed in, it is very likely they will be converted to float so that
they can stored in an `ndarray`. These warnings apply similarly to
`Series` since it internally leverages `ndarray`.

#### Parameters
arg : scalar, list, tuple, 1-d array, or Series
    Argument to be converted.
errors : {'ignore', 'raise', 'coerce'}, default 'raise'
    - If 'raise', then invalid parsing will raise an exception.
    - If 'coerce', then invalid parsing will be set as NaN.
    - If 'ignore', then invalid parsing will return the input.
downcast : str, default None
    Can be 'integer', 'signed', 'unsigned', or 'float'.
    If not None, and if the data has been successfully cast to a
    numerical dtype (or if the data was numeric to begin with),
    downcast that resulting data to the smallest numerical dtype
    possible according to the following rules:

    - 'integer' or 'signed': smallest signed int dtype (min.: np.int8)
    - 'unsigned': smallest unsigned int dtype (min.: np.uint8)
    - 'float': smallest float dtype (min.: np.float32)

    As this behaviour is separate from the core conversion to
    numeric values, any errors raised during the downcasting
    will be surfaced regardless of the value of the 'errors' input.

    In addition, downcasting will only occur if the size
    of the resulting data's dtype is strictly larger than
    the dtype it is to be cast to, so if none of the dtypes
    checked satisfy that specification, no downcasting will be
    performed on the data.

#### Returns
ret
    Numeric if parsing succeeded.
    Return type depends on input.  Series if Series, otherwise ndarray.

#### See Also
DataFrame.astype : Cast argument to a specified dtype.
to_datetime : Convert argument to datetime.
to_timedelta : Convert argument to timedelta.
numpy.ndarray.astype : Cast a numpy array to a specified type.
DataFrame.convert_dtypes : Convert dtypes.

#### Examples
Take separate series and convert to numeric, coercing when told to

>>> s = pd.Series(['1.0', '2', -3])
>>> pd.to_numeric(s)
0    1.0
1    2.0
2   -3.0
dtype: float64
>>> pd.to_numeric(s, downcast='float')
0    1.0
1    2.0
2   -3.0
dtype: float32
>>> pd.to_numeric(s, downcast='signed')
0    1
1    2
2   -3
dtype: int8
>>> s = pd.Series(['apple', '1.0', '2', -3])
>>> pd.to_numeric(s, errors='ignore')
0    apple
1      1.0
2        2
3       -3
dtype: object
>>> pd.to_numeric(s, errors='coerce')
0    NaN
1    1.0
2    2.0
3   -3.0
dtype: float64

Downcasting of nullable integer and floating dtypes is supported:

>>> s = pd.Series([1, 2, 3], dtype="Int64")
>>> pd.to_numeric(s, downcast="integer")
0    1
1    2
2    3
dtype: Int8
>>> s = pd.Series([1.0, 2.1, 3.0], dtype="Float64")
>>> pd.to_numeric(s, downcast="float")
0    1.0
1    2.1
2    3.0
dtype: Float32
File:      ~/anaconda3/lib/python3.11/site-packages/pandas/core/tools/numeric.py
Type:      function

In [23]:
df['nums_tonumeric'] = pd.to_numeric(df['nums'], errors='ignore')
df

,numbers,nums,colors,other_column,colors_upper,nums_tonumeric
0,#23,23,green,0,GREEN,23
1,#24,24,red,1,RED,24
2,#18,18,yellow,0,YELLOW,18
3,#14,14,orange,2,ORANGE,14
4,#12,NaN,purple,1,PURPLE,NaN
5,#10,XYZ,blue,0,BLUE,XYZ
6,#35,35,pink,2,PINK,35


**&rarr; As we could see in the docstring, I used** `errors='ignore'` **method**

Or, we can use the following way if we want all the data in the `nums_tonumeric` to be numeric. I filled the `NaN` values with 0 and changed string values into 0.

In [24]:
df_n = df

In [25]:
df_n['nums'] = df_n['nums'].fillna(0)
df_n['nums'][df_n['nums'] == 'XYZ'] = 0

/var/folders/bj/_3q81xv53xb91cdh9z0fl6bc0000gn/T/ipykernel_7256/3913110711.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_n['nums'][df_n['nums'] == 'XYZ'] = 0


In [26]:
df_n['nums']
    # I changed the values of df['nums'] that are not numeric, giving the value of '0'.

0    23
1    24
2    18
3    14
4     0
5     0
6    35
Name: nums, dtype: object

In [27]:
df_n['nums_tonumeric'] = pd.to_numeric(df_n['nums'])
df_n.head()

,numbers,nums,colors,other_column,colors_upper,nums_tonumeric
0,#23,23,green,0,GREEN,23
1,#24,24,red,1,RED,24
2,#18,18,yellow,0,YELLOW,18
3,#14,14,orange,2,ORANGE,14
4,#12,0,purple,1,PURPLE,0


### Exercise 4

We’d like you to use this data to answer the following questions.

- What is the average price of an item with chicken?  
- What is the average price of an item with steak?  
- Did chicken or steak produce more revenue (total)?  
- How many missing items are there in this dataset? How many missing
  items in each column?  


Before you will be able to do any of these things you will need to
make sure the `item_price` column has a numeric `dtype` (probably
float).

([back to text](#pd-cln-dir4))

#### Average price of an item with chicken

In [28]:
chipotle = chipotle.drop('Unnamed: 0', axis =1)
chipotle['price_numeric'] = chipotle["item_price"].str.replace("$", "")
chipotle['price_numeric'] = pd.to_numeric(chipotle['price_numeric'])
chipotle

/var/folders/bj/_3q81xv53xb91cdh9z0fl6bc0000gn/T/ipykernel_7256/1362230375.py:2: FutureWarning: The default value of regex will change from True to False in a future version. In addition, single character regular expressions will *not* be treated as literal strings when regex=True.
  chipotle['price_numeric'] = chipotle["item_price"].str.replace("$", "")


,order_id,quantity,item_name,choice_description,item_price,price_numeric
0,1,1,Chips and Fresh Tomato Salsa,NaN,$2.39,2.39
1,1,1,Izze,[Clementine],$3.39,3.39
2,1,1,Nantucket Nectar,[Apple],$3.39,3.39
3,1,1,Chips and Tomatillo-Green Chili Salsa,NaN,$2.39,2.39
4,2,2,Chicken Bowl,"[Tomatillo-Red Chili Salsa (Hot), [Black Beans...",$16.98,16.98
...,...,...,...,...,...,...
4617,1833,1,Steak Burrito,"[Fresh Tomato Salsa, [Rice, Black Beans, Sour ...",$11.75,11.75
4618,1833,1,Steak Burrito,"[Fresh Tomato Salsa, [Rice, Sour Cream, Cheese...",$11.75,11.75
4619,1834,1,Chicken Salad Bowl,"[Fresh Tomato Salsa, [Fajita Vegetables, Pinto...",$11.25,11.25
4620,1834,1,Chicken Salad Bowl,"[Fresh Tomato Salsa, [Fajita Vegetables, Lettu...",$8.75,8.75


In [29]:
chip_chicken = chipotle.loc[chipotle['item_name'].str.contains('Chicken')]
chip_chicken

,order_id,quantity,item_name,choice_description,item_price,price_numeric
4,2,2,Chicken Bowl,"[Tomatillo-Red Chili Salsa (Hot), [Black Beans...",$16.98,16.98
5,3,1,Chicken Bowl,"[Fresh Tomato Salsa (Mild), [Rice, Cheese, Sou...",$10.98,10.98
11,6,1,Chicken Crispy Tacos,"[Roasted Chili Corn Salsa, [Fajita Vegetables,...",$8.75,8.75
12,6,1,Chicken Soft Tacos,"[Roasted Chili Corn Salsa, [Rice, Black Beans,...",$8.75,8.75
13,7,1,Chicken Bowl,"[Fresh Tomato Salsa, [Fajita Vegetables, Rice,...",$11.25,11.25
...,...,...,...,...,...,...
4604,1828,1,Chicken Bowl,"[Fresh Tomato Salsa, [Rice, Black Beans, Chees...",$8.75,8.75
4615,1832,1,Chicken Soft Tacos,"[Fresh Tomato Salsa, [Rice, Cheese, Sour Cream]]",$8.75,8.75
4619,1834,1,Chicken Salad Bowl,"[Fresh Tomato Salsa, [Fajita Vegetables, Pinto...",$11.25,11.25
4620,1834,1,Chicken Salad Bowl,"[Fresh Tomato Salsa, [Fajita Vegetables, Lettu...",$8.75,8.75


In [30]:
chip_chicken['price_numeric'].mean()

10.133724358974359

#### Average price of an item with steak

In [31]:
chip_steak = chipotle.loc[chipotle['item_name'].str.contains('Steak')]
chip_steak

,order_id,quantity,item_name,choice_description,item_price,price_numeric
7,4,1,Steak Burrito,"[Tomatillo Red Chili Salsa, [Fajita Vegetables...",$11.75,11.75
8,4,1,Steak Soft Tacos,"[Tomatillo Green Chili Salsa, [Pinto Beans, Ch...",$9.25,9.25
9,5,1,Steak Burrito,"[Fresh Tomato Salsa, [Rice, Black Beans, Pinto...",$9.25,9.25
31,16,1,Steak Burrito,"[[Roasted Chili Corn Salsa (Medium), Fresh Tom...",$8.99,8.99
43,20,1,Steak Burrito,"[Fresh Tomato Salsa, [Rice, Pinto Beans, Chees...",$11.75,11.75
...,...,...,...,...,...,...
4572,1818,1,Steak Salad Bowl,"[Fresh Tomato Salsa, [Fajita Vegetables, Lettu...",$9.39,9.39
4607,1829,1,Steak Burrito,"[Tomatillo Green Chili Salsa, [Rice, Cheese, S...",$11.75,11.75
4610,1830,1,Steak Burrito,"[Fresh Tomato Salsa, [Rice, Sour Cream, Cheese...",$11.75,11.75
4617,1833,1,Steak Burrito,"[Fresh Tomato Salsa, [Rice, Black Beans, Sour ...",$11.75,11.75


In [32]:
chip_steak['price_numeric'].mean()

10.51888888888889

#### Producing More Revenue: chicken or steak (total)

In [33]:
TR_chicken = sum(chip_chicken['price_numeric'])
TR_steak = sum(chip_steak['price_numeric'])
print("Total revenue of chicken-containing items:", TR_chicken)
print("Total revenue of steak-containing items:", TR_steak)

print("")

if TR_chicken > TR_steak:
    print("Chicken-containing items produce more total revenue.")
elif TR_chicken < TR_steak:
    print("Steak-containing items produce more total revenue.")
else:
    print("Chicken or steak, ... Whatever!")

Total revenue of chicken-containing items: 15808.609999999922
Total revenue of steak-containing items: 7384.259999999973

Chicken-containing items produce more total revenue.


#### How many missing items are there in this dataset? How many missing items in each column?

In [34]:
chipotle.isnull().sum()

order_id                 0
quantity                 0
item_name                0
choice_description    1246
item_price               0
price_numeric            0
dtype: int64